<a href="https://colab.research.google.com/github/evkoff/DI-Bootcamp-Stage1/blob/main/Week16/Day5/Tutorial/W16D5_Mistral_colab_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Exploring Mistral 7B in Colab
This interactive Google Colab tutorial walks you through loading, profiling, exploring one Mistral AI model —**Mistral 7B** its performance and quality trade-offs.

## 1️⃣ Setup
Install required libraries and enable GPU runtime. **Make sure you go to Runtime → Change runtime type → Hardware accelerator → GPU and re-started.**

In [ ]:
!pip install -q transformers accelerate bitsandbytes

import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 16.0 MB/s eta 0:00:00


## 2️⃣ Load Models
Load Mistral 7B with 8-bit quantization.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# 8-bit quantization config
bnb_cfg = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,     # optional: tweak for your use-case
)

# Load Mistral 7B (8-bit)
tokenizer_7b = AutoTokenizer.from_pretrained("mayanklohani19/Mistral-7B-Mistral-7B")
model_7b = AutoModelForCausalLM.from_pretrained(
    "mayanklohani19/Mistral-7B-Mistral-7B",
    quantization_config=bnb_cfg,    # ← new arg
    device_map="auto",
)



config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.51M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/11.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/147 [00:00<?, ?it/s]

## 3️⃣ Simple Generation & Profiling
Define a helper to measure latency and memory usage during generation.

In [ ]:
def profile(model, tokenizer, prompt, max_new_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    elapsed = time.time() - start
    peak_mem = torch.cuda.max_memory_allocated() / (1024**2)  # MiB
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text, elapsed, peak_mem

prompt = "In a distant future, AI and humans collaborate to"
res_7b = profile(model_7b, tokenizer_7b, prompt)

print("Mistral 7B:", res_7b)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Mistral 7B: ('In a distant future, AI and humans collaborate to are are are are are are are are are are....... are. are with with with with...\n...\n.\n\n the\n the the the the the the the\n the\n the the.', 6.179889678955078, 4478.08642578125)


# Bonus

## 4️⃣ Summarization Task
Load summarization pipelines and compare summaries.

In [ ]:
article = """
[Paste a 200-word article here...]
"""

summarizer_7b = pipeline(
    "summarization",
    model=model_7b,
    tokenizer=tokenizer_7b,
    device=model_7b.device.index
)


sum_7b = summarizer_7b(article, max_length=30, min_length=10, do_sample=False)[0]["summary_text"]
print("7B Summary:", sum_7b)


## 5️⃣ Analysis & Discussion
- **Performance:** Compare speed (seconds) and memory (MiB).
- **Quality:** Evaluate fluency and conciseness of generated text.
- **Use-Case Fit:** When to choose 7B vs. Large 2 based on resource constraints and accuracy needs.